# MLP pt2

Same data pipeline and shared-vocabulary event tokenization as `mlp.ipynb`. One change from pt1, in isolation: a deeper network (`37 -> 96 -> 96 -> 96 -> 107` instead of `37 -> 80 -> 107`), staying with raw-scalar categorical inputs and the shared 107-wide output vocab for now — those are queued as separate follow-up changes so we can attribute any improvement to depth/width alone, not a bundle of changes at once.

In [1]:
import warnings
from statsbombpy.api_client import NoAuthWarning
from socceraction.data.statsbomb import StatsBombLoader
import pandas as pd
from statsbombpy import sb
import numpy as np
import json
import socceraction.spadl as spadl

warnings.filterwarnings("ignore", category=NoAuthWarning)
warnings.filterwarnings("ignore", message="credentials were not supplied")

pd.set_option("display.max_columns", None)  # Show all columns

In [2]:
df = pd.read_csv('data/masked_df.csv', index_col=0)

In [ ]:
df.describe()

In [ ]:
df.loc[df["target"] == -2, "target"] = 106

In [ ]:
with open("data/event_masks.json") as f:
    masks = json.load(f)

with open("data/category_lookup.json") as f:
    lookup = json.load(f)

In [ ]:
lookup = {
    col: {int(k): v for k, v in mapping.items()}
    for col, mapping in lookup.items()
}

In [ ]:
masks = {
    col: [106 if v == -2 else v for v in vals]
    for col, vals in masks.items()
}

In [ ]:
ml_df = df

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline
import random

random.seed(42)
g = torch.Generator().manual_seed(42)  # shared generator for every torch random draw below

In [ ]:
ml_events = ml_df[['game_id']].drop_duplicates()

In [ ]:
events_shuffled = ml_events.sample(frac=1, random_state=42)
n1 = int(0.8 * len(events_shuffled))
n2 = int(0.9 * len(events_shuffled))

In [ ]:
df_shuffled1 = pd.merge(
    events_shuffled.iloc[:n1],
    ml_df,
    left_on=["game_id"],
    right_on=["game_id"],
    how="left",
)
df_shuffled2 = pd.merge(
    events_shuffled.iloc[n1:n2],
    ml_df,
    left_on=["game_id"],
    right_on=["game_id"],
    how="left",
)
df_shuffled3 = pd.merge(
    events_shuffled.iloc[n2:],
    ml_df,
    left_on=["game_id"],
    right_on=["game_id"],
    how="left",
)

In [ ]:
def build_dataset(df):
    X = df[
        [
            "is_home",
            "event",
            "start_x",
            "start_y",
            "time_elapsed",
            "duration",
            "result",
            "period_id",
            "minute",
            "second",
            "home_goal_count",
            "away_goal_count",
            "home_red_count",
            "away_red_count",
            "home_yellow_count",
            "away_yellow_count",
            "is_home-1",
            "event-1",
            "start_x-1",
            "start_y-1",
            "time_elapsed-1",
            "duration-1",
            "result-1",
            "is_home-2",
            "event-2",
            "start_x-2",
            "start_y-2",
            "time_elapsed-2",
            "duration-2",
            "result-2",
            "is_home-3",
            "event-3",
            "start_x-3",
            "start_y-3",
            "time_elapsed-3",
            "duration-3",
            "result-3",
            "target",
        ]
    ]
    X = X.drop(['target'], axis=1)
    X = torch.tensor(X.values).float()

    Y = df['target']
    Y = torch.tensor(Y.values).float()
    Y = F.one_hot(Y.long(), num_classes=107).float()

    return X, Y

In [ ]:
Xtr, Ytr = build_dataset(df_shuffled1)
Xval, Yval = build_dataset(df_shuffled2)
Xte, Yte = build_dataset(df_shuffled3)

In [ ]:
Xtr.shape, Xval.shape, Xte.shape

## Baseline: zeroed inputs

Same change as the real model below (deeper network) but every input feature
is zeroed out for every batch, so the model can only fit the marginal
distribution of the target. This is the floor the real model needs to beat —
if the deeper network can't clear this by roughly the same margin pt1's did,
depth/width wasn't the bottleneck.

In [ ]:
n_hidden = 96
vocab_size = len(Ytr[0])

g_baseline = torch.Generator().manual_seed(42)

W1_baseline = torch.randn((Xtr.shape[1], n_hidden), generator=g_baseline) * 0.001
b1_baseline = torch.randn(n_hidden, generator=g_baseline) * 0
W2_baseline = torch.randn((n_hidden, n_hidden), generator=g_baseline) * 0.001
b2_baseline = torch.randn(n_hidden, generator=g_baseline) * 0
W3_baseline = torch.randn((n_hidden, n_hidden), generator=g_baseline) * 0.001
b3_baseline = torch.randn(n_hidden, generator=g_baseline) * 0
W4_baseline = torch.randn((n_hidden, vocab_size), generator=g_baseline) * 0.001
b4_baseline = torch.randn(vocab_size, generator=g_baseline) * 0

baseline_parameters = [
    W1_baseline, b1_baseline,
    W2_baseline, b2_baseline,
    W3_baseline, b3_baseline,
    W4_baseline, b4_baseline,
]

for p in baseline_parameters:
    p.requires_grad = True

baseline_lossi = []

In [ ]:
baseline_steps = 100000
baseline_batch_size = 512
baseline_lr = 0.1


for i in range(baseline_steps):
    ix = torch.randint(0, Xtr.shape[0], (baseline_batch_size,), generator=g_baseline)
    Xb, Yb = torch.zeros_like(Xtr[ix]), Ytr[ix]
    
    #forward pass
    
    h1preact = Xb @ W1_baseline + b1_baseline
    h1 = torch.relu(h1preact)
    
    h2preact = h1 @ W2_baseline + b2_baseline
    h2 = torch.relu(h2preact)
    
    h3preact = h2 @ W3_baseline + b3_baseline
    h3 = torch.relu(h3preact)
    
    logits = h3 @ W4_baseline + b4_baseline
    
    loss = F.binary_cross_entropy_with_logits(logits, Yb)
    
    baseline_lossi.append(loss.item())
    
    # backward pass
    
    for p in baseline_parameters:
        p.grad = None
        
    loss.backward()
    
    for p in baseline_parameters:
        p.data += -baseline_lr * p.grad

    if i % 10000 == 0:
        print(i, loss.item())
    
#baseline_lossi

In [ ]:
plt.plot(range(len(baseline_lossi)), baseline_lossi)

In [ ]:
np.mean(baseline_lossi[:20]), np.mean(baseline_lossi[-20:])

In [ ]:
with torch.no_grad():
    val_h1 = torch.relu(torch.zeros_like(Xval) @ W1_baseline + b1_baseline)
    val_h2 = torch.relu(val_h1 @ W2_baseline + b2_baseline)
    val_h3 = torch.relu(val_h2 @ W3_baseline + b3_baseline)
    logits = val_h3 @ W4_baseline + b4_baseline
    baseline_val_loss = F.binary_cross_entropy_with_logits(logits, Yval)

baseline_val_loss.item()

In [ ]:
event_cols = ['is_home', 'event', 'start_x', 'start_y', 'time_elapsed', 'duration', 'result']

with torch.no_grad():
    start = torch.zeros((1, Xtr.shape[1]))

    for i, field in enumerate(event_cols):
        mask = masks[field]

        h1 = torch.relu(start @ W1_baseline + b1_baseline)
        h2 = torch.relu(h1 @ W2_baseline + b2_baseline)
        h3 = torch.relu(h2 @ W3_baseline + b3_baseline)
        logits = h3 @ W4_baseline + b4_baseline
        restricted_logits = logits[:, mask]

        ix = restricted_logits.argmax(dim=1)
        mask = torch.tensor(mask)
        value = mask[ix].item()
        value = -2 if value == 106 else value

        start[0, i] = value

        if field in lookup:
            print(field, value, lookup[field][value])
        else:
            print(field, value)

## Real model

Same 3-hidden-layer architecture as the baseline above, trained on the real
inputs. LR held constant at 0.1 for the full run — same as pt1's first
stage, and same as the baseline above. (The adaptive ×1.5/÷1.5-on-any-improvement
rule tried earlier diverged to NaN by step 700K — even a 10K-step smoke test
showed LR blowing up 38x in the first few checkpoints, since loss improves on
almost every early checkpoint with nothing capping the climb. Dropped in favor
of just holding LR fixed.)

In [ ]:
n_hidden = 96
vocab_size = len(Ytr[0])

W1 = torch.randn((Xtr.shape[1], n_hidden), generator=g) * 0.001
b1 = torch.randn(n_hidden, generator=g) * 0
W2 = torch.randn((n_hidden, n_hidden), generator=g) * 0.001
b2 = torch.randn(n_hidden, generator=g) * 0
W3 = torch.randn((n_hidden, n_hidden), generator=g) * 0.001
b3 = torch.randn(n_hidden, generator=g) * 0
W4 = torch.randn((n_hidden, vocab_size), generator=g) * 0.001
b4 = torch.randn(vocab_size, generator=g) * 0

parameters = [W1, b1, W2, b2, W3, b3, W4, b4]

for p in parameters:
    p.requires_grad = True

lossi = []

In [ ]:
max_steps = 1000000
batch_size = 512
checkpoint_every = 50000

lr = 0.1


for i in range(max_steps):
    ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
    Xb, Yb = Xtr[ix], Ytr[ix]
    
    #forward pass
    
    h1preact = Xb @ W1 + b1
    h1 = torch.relu(h1preact)
    
    h2preact = h1 @ W2 + b2
    h2 = torch.relu(h2preact)
    
    h3preact = h2 @ W3 + b3
    h3 = torch.relu(h3preact)
    
    logits = h3 @ W4 + b4
    
    loss = F.binary_cross_entropy_with_logits(logits, Ytr[ix])
    
    # backward pass
    
    for p in parameters:
        p.grad = None
        
    loss.backward()
    
    for p in parameters:
        p.data += -lr * p.grad

    if i % checkpoint_every == 0:
        with torch.no_grad():
            val_h1 = torch.relu(Xval @ W1 + b1)
            val_h2 = torch.relu(val_h1 @ W2 + b2)
            val_h3 = torch.relu(val_h2 @ W3 + b3)
            val_logits = val_h3 @ W4 + b4
            val_loss = F.binary_cross_entropy_with_logits(val_logits, Yval)

        lossi.append(val_loss.item())
        print(i, lr, val_loss.item())

with torch.no_grad():
    val_h1 = torch.relu(Xval @ W1 + b1)
    val_h2 = torch.relu(val_h1 @ W2 + b2)
    val_h3 = torch.relu(val_h2 @ W3 + b3)
    val_logits = val_h3 @ W4 + b4
    val_loss = F.binary_cross_entropy_with_logits(val_logits, Yval)

lossi.append(val_loss.item())
print(max_steps, lr, val_loss.item())

#lossi

In [ ]:
plt.plot(range(0, len(lossi) * checkpoint_every, checkpoint_every), lossi)

In [ ]:
lossi[0], lossi[-1]

In [ ]:
@torch.no_grad()  # this decorator disables gradient tracking
def split_loss(split):
    x, y = {
        "train": (Xtr, Ytr),
        "val": (Xval, Yval),
        "test": (Xte, Yte),
    }[
        split
    ]  # (N, block_size, n_embd)

    h1 = torch.relu(x @ W1 + b1)
    h2 = torch.relu(h1 @ W2 + b2)
    h3 = torch.relu(h2 @ W3 + b3)

    logits = h3 @ W4 + b4  # (N, vocab_size)
    loss = F.binary_cross_entropy_with_logits(logits, y)
    print(split, loss.item())


def split_accuracy(split):
    x, y = {
        "train": (Xtr, Ytr),
        "val": (Xval, Yval),
        "test": (Xte, Yte),
    }[
        split
    ]  # (N, block_size, n_embd)

    h1 = torch.relu(x @ W1 + b1)
    h2 = torch.relu(h1 @ W2 + b2)
    h3 = torch.relu(h2 @ W3 + b3)

    logits = h3 @ W4 + b4
    ix = logits.argmax(dim=1)
    was_accurate = y[torch.arange(y.shape[0]), ix]

    accuracy = was_accurate.sum() /len(was_accurate)
    print(split, accuracy.item())

In [ ]:
event_cols = ['is_home', 'event', 'start_x', 'start_y', 'time_elapsed', 'duration', 'result']
acc_fields = ['is_home', 'event', 'result']


@torch.no_grad()
def split_accuracy_by_field(split):
    x, y, step = {
        "train": (Xtr, Ytr, df_shuffled1['step'].values),
        "val": (Xval, Yval, df_shuffled2['step'].values),
        "test": (Xte, Yte, df_shuffled3['step'].values),
    }[split]

    step = torch.tensor(step)

    h1 = torch.relu(x @ W1 + b1)
    h2 = torch.relu(h1 @ W2 + b2)
    h3 = torch.relu(h2 @ W3 + b3)

    logits = h3 @ W4 + b4
    ix = logits.argmax(dim=1)
    true = y.argmax(dim=1)

    for i, field in enumerate(event_cols):
        mask = step == i
        pred_i, true_i = ix[mask], true[mask]

        if field in acc_fields:
            was_accurate = pred_i == true_i
            accuracy = was_accurate.sum() / len(was_accurate)
            print(split, field, 'accuracy', accuracy.item())
        else:
            pred_val = torch.where(pred_i == 106, torch.tensor(-2), pred_i)
            true_val = torch.where(true_i == 106, torch.tensor(-2), true_i)
            mae = (pred_val - true_val).abs().float().mean()
            print(split, field, 'MAE', mae.item())

In [ ]:
@torch.no_grad()
def predict_events(start):
    start = start.clone()

    for i, field in enumerate(event_cols):
        mask = masks[field]

        h1 = torch.relu(start @ W1 + b1)
        h2 = torch.relu(h1 @ W2 + b2)
        h3 = torch.relu(h2 @ W3 + b3)

        logits = h3 @ W4 + b4
        restricted_logits = logits[:, mask]

        ix = restricted_logits.argmax(dim=1)
        mask = torch.tensor(mask)
        value = mask[ix]
        value = torch.where(value == 106, torch.tensor(-2), value)

        start[:, i] = value.float()

    return start[:, :len(event_cols)]

In [ ]:
def autoregressive_accuracy(split):
    df = {"train": df_shuffled1, "val": df_shuffled2, "test": df_shuffled3}[split]

    gt = df[df.step == 6].copy()
    gt = gt.drop(columns=['result']).rename(columns={'target': 'result'})
    gt = gt.set_index(['game_id', 'index'])

    boundary_event_codes = {0, 18, 21}
    clean = (
        (~gt['event'].isin(boundary_event_codes))
        & (gt['is_home-1'] != -2)
        & (gt['is_home-2'] != -2)
        & (gt['is_home-3'] != -2)
    )
    gt = gt[clean]

    start = df[df.step == 0].set_index(['game_id', 'index']).loc[gt.index]

    model_input_cols = [
        "is_home", "event", "start_x", "start_y", "time_elapsed", "duration", "result",
        "period_id", "minute", "second", "home_goal_count", "away_goal_count",
        "home_red_count", "away_red_count", "home_yellow_count", "away_yellow_count",
        "is_home-1", "event-1", "start_x-1", "start_y-1", "time_elapsed-1", "duration-1", "result-1",
        "is_home-2", "event-2", "start_x-2", "start_y-2", "time_elapsed-2", "duration-2", "result-2",
        "is_home-3", "event-3", "start_x-3", "start_y-3", "time_elapsed-3", "duration-3", "result-3",
    ]
    x = torch.tensor(start[model_input_cols].values).float()

    pred = predict_events(x)

    for i, field in enumerate(event_cols):
        pred_i = pred[:, i]
        true_i = torch.tensor(gt[field].values).float()

        if field in acc_fields:
            accuracy = (pred_i == true_i).float().mean()
            print(split, field, 'accuracy', accuracy.item())
        else:
            mae = (pred_i - true_i).abs().mean()
            print(split, field, 'MAE', mae.item())

In [ ]:
autoregressive_accuracy('train')

In [ ]:
split_loss("train")
split_loss("val")

In [ ]:
split_accuracy_by_field('train')

## Sampling

Same restricted-softmax sampling procedure as pt1, forward pass updated for
the deeper network.

In [ ]:
def sample_token(start):
    i, field = next((i, col) for i, col in enumerate(event_cols) if start[0, i] == -1)

    mask = masks[field]

    h1 = torch.relu(start @ W1 + b1)
    h2 = torch.relu(h1 @ W2 + b2)
    h3 = torch.relu(h2 @ W3 + b3)

    logits = h3 @ W4 + b4
    restricted_logits = logits[:, mask]

    probs = F.softmax(restricted_logits, dim=1)
    ix = probs.multinomial(num_samples=1, replacement=True, generator=g)
    sample = mask[ix.item()]

    value = -2 if sample == 106 else sample

    return value

In [ ]:
def sample_event(start):
    start = start.clone()

    ix, field = next((i, col) for i, col in enumerate(event_cols) if start[0, i] == -1)

    for i in range(ix, len(event_cols)):
        val = sample_token(start)

        start[0, i] = val

    start = start[0]

    event_token = {
        "is_home": (start[0].item(), lookup["is_home"][start[0].item()]),
        "event": (start[1].item(), lookup["event"][start[1].item()]),
        "start_x": start[2].item(),
        "start_y": start[3].item(),
        "time_elapsed": start[4].item(),
        "duration": start[5].item(),
        "result": (start[6].item(), lookup["result"][start[6].item()]),
    }

    return event_token

Sample a fresh event from scratch, given only the last three events as
context (everything for the current event masked to `-1`).

In [ ]:
sample_start = df_shuffled2[df_shuffled2['step'] == 0].iloc[[0]]
model_input_cols = [
    "is_home", "event", "start_x", "start_y", "time_elapsed", "duration", "result",
    "period_id", "minute", "second", "home_goal_count", "away_goal_count",
    "home_red_count", "away_red_count", "home_yellow_count", "away_yellow_count",
    "is_home-1", "event-1", "start_x-1", "start_y-1", "time_elapsed-1", "duration-1", "result-1",
    "is_home-2", "event-2", "start_x-2", "start_y-2", "time_elapsed-2", "duration-2", "result-2",
    "is_home-3", "event-3", "start_x-3", "start_y-3", "time_elapsed-3", "duration-3", "result-3",
]
sample_x = torch.tensor(sample_start[model_input_cols].values).float()

sample_event(sample_x)

## Next up

Still queued, one at a time:

1. One-hot encode the categorical inputs (`is_home`, `event`, `result`, and
   their `-1`/`-2`/`-3` lags) instead of feeding them in as raw scalar codes
   — boundary values get their own explicit one-hot slot rather than an
   all-zero vector.
2. Disjoint per-field output vocabulary: one linear output layer sized to
   the sum of each field's own cardinality (using `event_masks.json`)
   instead of one shared 107-wide space, so a class index means one thing
   only. Loss stays a plain elementwise BCE over the full output — no
   per-row masking needed, since the model has no reason to activate
   another field's slice and the restricted sampler already prevents
   nonsensical sampled values regardless.

## Persist

Model checkpoints only — the train/val/test splits are identical to pt1's
(same seed, same underlying data), so `data/train_set.csv` /
`data/val_set.csv` / `data/test_set.csv` from `mlp.ipynb` are still valid
and don't need to be rewritten here.

In [ ]:
baseline_checkpoint = {
    "W1": W1_baseline, "b1": b1_baseline,
    "W2": W2_baseline, "b2": b2_baseline,
    "W3": W3_baseline, "b3": b3_baseline,
    "W4": W4_baseline, "b4": b4_baseline,
    "lossi": baseline_lossi,
    "baseline_steps": baseline_steps,
    "seed": 42,
}
torch.save(baseline_checkpoint, "data/mlp_pt2_baseline_checkpoint.pt")

In [ ]:
checkpoint = {
    "W1": W1, "b1": b1,
    "W2": W2, "b2": b2,
    "W3": W3, "b3": b3,
    "W4": W4, "b4": b4,
    "lossi": lossi,
    "max_steps": max_steps,
    "seed": 42,
}
torch.save(checkpoint, "data/mlp_pt2_checkpoint.pt")